In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"     # agama
os.environ["MKL_NUM_THREADS"] = "1"     # numpy, scipy
os.environ["OPENBLAS_NUM_THREADS"] = "1"    # numpy
os.environ["NUMEXPR_NUM_THREADS"] = "1"     # pandas
import random
import agama
import torch 
import numpy as np
from scipy import integrate
from astropy import units as u

from sbi.utils import BoxUniform
from sbi.inference import SNLE, simulate_for_sbi, prepare_for_sbi
from sbi.utils import likelihood_nn

from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import pickle
import matplotlib.pyplot as plt
from galaxy_generation import generate_galaxy_multiple
# from python_scripts.prior_generation import generate_prior

agama.setRandomSeed(13)
torch.manual_seed(13)
np.random.seed(13)
random.seed(13)

torch.set_num_threads(4)

In [18]:
# Galaxy
log_p_0 = 7
log_r_s = 0
gamma = 0
r_star_div_r_s = 0.2
r_star = r_star_div_r_s * 10 ** log_r_s

galaxy = np.expand_dims(np.array([log_p_0, log_r_s, gamma, r_star_div_r_s]), axis=0).astype(np.float32)

In [3]:
galaxy

array([[7. , 0. , 1. , 0.2]], dtype=float32)

In [4]:
df = generate_galaxy_multiple(torch.tensor(galaxy), [100], 1)

pd.DataFrame(df).to_csv("x_o_cusp.csv", header=None, index=None)

## Contour

In [19]:
with open("inference_model_9_core.pkl", "rb") as file:
    inference = pickle.load(file)

In [20]:
df = inference._neural_net.sample(100_000, context=galaxy).squeeze().detach().numpy()

In [21]:
pd.DataFrame(df).to_csv("contour_samples_model_9_core", header=None, index=None)